# Bulk Deconvolution — Cell-Type-Aware Subtyping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/04_deconvolution.ipynb)

**What this does:** Estimates cell-type proportions from bulk RNA-seq using a single-cell reference, then combines proportions with pathway scores so subtypes become both pathway-defined AND cell-type-aware.

**Algorithm:** Non-Negative Least Squares (NNLS) — the same mathematical foundation as CIBERSORT. For each bulk sample, solves: `bulk ≈ reference × proportions` subject to `proportions ≥ 0`.

**Prerequisites:** [02_expression_scoring.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/02_expression_scoring.ipynb)

In [ ]:
# Install pathway-subtyping
!pip install -q pathway-subtyping==0.3.0

import pathway_subtyping
print(f"pathway-subtyping v{pathway_subtyping.__version__}")

## 1. Generate Synthetic Bulk Data

We'll create synthetic bulk expression from known cell-type proportions. This lets us verify that deconvolution recovers the true proportions.

In [ ]:
from pathway_subtyping import generate_synthetic_bulk
import numpy as np
import pandas as pd

# First, create a simple reference profile (5 cell types × 200 genes)
np.random.seed(42)
cell_types = ["Neuron", "Astrocyte", "Microglia", "Oligodendrocyte", "Endothelial"]
n_genes = 200
gene_names = [f"Gene_{i}" for i in range(n_genes)]

# Each cell type has a distinct expression signature
reference_profile = pd.DataFrame(
    np.abs(np.random.randn(len(cell_types), n_genes) * 2 + 5),
    index=cell_types,
    columns=gene_names,
)
# Add cell-type-specific marker genes
for i, ct in enumerate(cell_types):
    marker_start = i * 20
    reference_profile.iloc[i, marker_start:marker_start + 20] *= 5

print(f"Reference profile: {reference_profile.shape[0]} cell types × {reference_profile.shape[1]} genes")
print(f"Cell types: {list(reference_profile.index)}")

# Generate synthetic bulk from known proportions
bulk_expr, true_proportions, subtype_labels = generate_synthetic_bulk(
    reference_profile,
    n_samples=100,
    n_subtypes=3,
    noise_level=0.1,
    seed=42,
)

print(f"\nSynthetic bulk: {bulk_expr.shape[0]} samples × {bulk_expr.shape[1]} genes")
print(f"True proportions: {true_proportions.shape}")
print(f"Subtypes planted: {len(set(subtype_labels))}")

print(f"\nMean cell-type proportions per subtype:")
prop_df = true_proportions.copy()
prop_df["subtype"] = subtype_labels
print(prop_df.groupby("subtype").mean().round(3).to_string())

## 2. Deconvolve Bulk Expression

Run NNLS deconvolution to estimate cell-type proportions from the bulk data.

In [ ]:
from pathway_subtyping import deconvolve_bulk, DeconvolutionMethod

result = deconvolve_bulk(
    bulk_expr,
    reference_profile,
    method=DeconvolutionMethod.NNLS,
    seed=42,
)

print("Deconvolution Result")
print(f"  Method:     {result.method.value}")
print(f"  Samples:    {result.n_samples}")
print(f"  Cell types: {result.n_cell_types} ({', '.join(result.reference_cell_types)})")
print(f"\nQuality Report:")
print(f"  Genes in bulk:      {result.quality_report.n_genes_bulk}")
print(f"  Genes in reference: {result.quality_report.n_genes_reference}")
print(f"  Genes shared:       {result.quality_report.n_genes_shared}")
print(f"  Gene coverage:      {result.quality_report.gene_coverage:.1%}")
print(f"  Usable:             {result.quality_report.is_usable}")

print(f"\nEstimated proportions (first 5 samples):")
result.cell_type_proportions.head()

## 3. Evaluate Recovery

Compare estimated proportions to ground truth. High correlation means the deconvolution is working well.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(cell_types), figsize=(20, 4))

for ax, ct in zip(axes, cell_types):
    true_vals = true_proportions[ct].values
    est_vals = result.cell_type_proportions[ct].values
    corr = np.corrcoef(true_vals, est_vals)[0, 1]

    ax.scatter(true_vals, est_vals, s=20, alpha=0.6)
    ax.plot([0, 1], [0, 1], "r--", lw=1, alpha=0.5)
    ax.set_title(f"{ct}\nr={corr:.3f}")
    ax.set_xlabel("True")
    ax.set_ylabel("Estimated")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

plt.suptitle("Deconvolution Recovery: Estimated vs True Proportions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Overall correlation
all_true = true_proportions.values.flatten()
all_est = result.cell_type_proportions.values.flatten()
print(f"Overall correlation: r = {np.corrcoef(all_true, all_est)[0,1]:.3f}")

## 4. Combine with Pathway Scores

Merge cell-type proportions with pathway scores into a unified feature matrix. The `proportion_weight` parameter controls the relative influence of cell-type information.

In [ ]:
from pathway_subtyping import (
    combine_features,
    SimulationConfig,
    generate_synthetic_data,
    run_clustering,
    ClusteringAlgorithm,
)
from sklearn.metrics import adjusted_rand_score

# Generate pathway scores for the same cohort
sim = generate_synthetic_data(SimulationConfig(
    n_samples=100,
    n_pathways=8,
    n_genes_per_pathway=20,
    n_subtypes=3,
    effect_size=1.2,
    noise_level=1.0,
    seed=42,
))

# Combine pathway scores + cell-type proportions
combined = combine_features(
    sim.pathway_scores,
    result.cell_type_proportions,
    proportion_weight=0.3,  # 30% cell-type, 70% pathway
)

print(f"Pathway scores only:  {sim.pathway_scores.shape}")
print(f"Cell-type proportions: {result.cell_type_proportions.shape}")
print(f"Combined features:    {combined.shape}")
print(f"\nCombined columns: {list(combined.columns)}")

# Cluster: pathway-only vs combined
cl_pathway = run_clustering(sim.pathway_scores.values, n_clusters=3, seed=42)
cl_combined = run_clustering(combined.values, n_clusters=3, seed=42)

ari_pathway = adjusted_rand_score(sim.true_labels, cl_pathway.labels)
ari_combined = adjusted_rand_score(sim.true_labels, cl_combined.labels)

print(f"\nClustering Comparison:")
print(f"  Pathway-only ARI:  {ari_pathway:.3f}  (Sil={cl_pathway.silhouette:.3f})")
print(f"  Combined ARI:      {ari_combined:.3f}  (Sil={cl_combined.silhouette:.3f})")

## 5. Visualize Combined Subtypes

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, scores, cl) in zip(axes, [
    ("Pathway Scores Only", sim.pathway_scores.values, cl_pathway),
    ("Pathway + Cell-Type", combined.values, cl_combined),
]):
    pca = PCA(n_components=2, random_state=42)
    X = pca.fit_transform(scores)

    for label in sorted(set(cl.labels)):
        mask = cl.labels == label
        ax.scatter(X[mask, 0], X[mask, 1], label=f"Cluster {label}", s=30, alpha=0.7)

    ari = adjusted_rand_score(sim.true_labels, cl.labels)
    ax.set_title(f"{title}\nARI={ari:.3f}")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
    ax.legend(fontsize=8)

plt.suptitle("Effect of Adding Cell-Type Proportions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Using Your Own Data

```python
from pathway_subtyping import build_reference_profile, deconvolve_bulk, combine_features
import scanpy as sc

# Load single-cell reference
adata = sc.read_h5ad("reference_scrnaseq.h5ad")
sc_expr = pd.DataFrame(adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)

# Build reference profile from single-cell data
reference = build_reference_profile(
    sc_expr,
    cell_type_labels=adata.obs["cell_type"].values,
    min_cells_per_type=5,
)

# Deconvolve your bulk data
bulk = pd.read_csv("bulk_expression.csv", index_col=0)
result = deconvolve_bulk(bulk, reference, seed=42)
print(result.format_report())

# Combine with pathway scores
combined = combine_features(pathway_scores, result.cell_type_proportions, proportion_weight=0.3)
```

## Next Steps

- **Multi-omic fusion:** [03_multi_omic_fusion.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/03_multi_omic_fusion.ipynb) — fuse VCF + expression
- **Visualization:** [05_visualization.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/05_visualization.ipynb) — interactive reports
- **API reference:** [Deconvolution API](https://github.com/topmist-admin/pathway-subtyping-framework/blob/main/docs/api/deconvolution.md)

---
*Built with [pathway-subtyping](https://pypi.org/project/pathway-subtyping/). Disease-agnostic. Open source.*